###
**==============================================================================**
# Task: 01_data_cleaning.py
# Author / Task Owner: Fatima Malik
# Sprint: Week 02 — Data Cleaning & Integration
# Project: CadetX Heavy Supplier, Inventory & Warehouse Analytics
### ==============================================================================

In [4]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

In [5]:
# Load core relational datasets
branches_df = pd.read_csv('/content/branches.csv')
customers_df = pd.read_csv('/content/customers.csv')
inventory_df = pd.read_csv('/content/inventory_master.csv')
invoices_df = pd.read_csv('/content/invoices.csv')
payments_df = pd.read_csv('/content/payments.csv')
products_df = pd.read_csv('/content/products.csv')
purchase_orders_header_df = pd.read_csv('/content/purchase_orders_header.csv')
purchase_orders_lines_df = pd.read_csv('/content/purchase_orders_lines.csv')
sales_orders_header_df = pd.read_csv('/content/sales_orders_header.csv')
sales_orders_lines_df = pd.read_csv('/content/sales_orders_lines.csv')
stock_ledger_df = pd.read_csv('/content/stock_ledger.csv')
suppliers_df = pd.read_csv('/content/suppliers.csv')

In [3]:
  # Dictionary to manage all datasets efficiently
datasets = {
    'branches': branches_df,
    'customers': customers_df,
    'inventory': inventory_df,
    'invoices': invoices_df,
    'payments': payments_df,
    'products': products_df,
    'purchase_headers': purchase_orders_header_df,
    'purchase_lines': purchase_orders_lines_df,
    'sales_headers': sales_orders_header_df,
    'sales_lines': sales_orders_lines_df,
    'stock_ledger': stock_ledger_df,
    'suppliers': suppliers_df
}

### **1. Standardise column names**

In [7]:
def standardise_column_names(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    return df

# Apply to all datasets

for name in datasets:

    datasets[name] = standardise_column_names(
        datasets[name]
    )

print("Column names standardised.")

Column names standardised.


### **2. Update the DataFrame variables**

In [8]:
branches_df = datasets['branches']
customers_df = datasets['customers']
inventory_df = datasets['inventory']
invoices_df = datasets['invoices']
payments_df = datasets['payments']
products_df = datasets['products']

purchase_orders_header_df = datasets['purchase_headers']
purchase_orders_lines_df = datasets['purchase_lines']

sales_orders_header_df = datasets['sales_headers']
sales_orders_lines_df = datasets['sales_lines']

stock_ledger_df = datasets['stock_ledger']
suppliers_df = datasets['suppliers']

## **3. Check the Columns after Standardisation**

In [9]:
for name, df in datasets.items():

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print(df.columns.tolist())


BRANCHES
['branch_id', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index']

CUSTOMERS
['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']

INVENTORY
['product_id', 'branch_id', 'opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock', 'warehouse_bin']

INVOICES
['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']

PAYMENTS
['payment_id', 'invoice_id', 'payment_date', 'payment_amount', 'payment_method']

PRODUCTS
['product_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'uni

### **4. Missing-value Analysis**

In [10]:
missing_summary = []

for name, df in datasets.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:

            missing_summary.append({
                'Dataset': name,
                'Column': column,
                'Missing Count': missing_count,
                'Missing %': round(
                    missing_count / len(df) * 100,
                    2
                )
            })

missing_df = pd.DataFrame(missing_summary)

if not missing_df.empty:
    display(
        missing_df.sort_values(
            'Missing %',
            ascending=False
        )
    )
else:
    print("No missing values found.")

,Dataset,Column,Missing Count,Missing %
0,purchase_headers,received_date,2370,9.88


### **5. Duplicate Analysis**

In [11]:
duplicate_summary = []

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        'Dataset': name,
        'Rows': len(df),
        'Duplicate Rows': duplicate_count,
        'Duplicate %': round(
            duplicate_count / len(df) * 100,
            2
        )
    })

duplicate_df = pd.DataFrame(
    duplicate_summary
)

display(duplicate_df)

,Dataset,Rows,Duplicate Rows,Duplicate %
0,branches,6,0,0.0
1,customers,500,0,0.0
2,inventory,180,0,0.0
3,invoices,18033,0,0.0
4,payments,19257,0,0.0
5,products,30,0,0.0
6,purchase_headers,24000,0,0.0
7,purchase_lines,155495,0,0.0
8,sales_headers,20000,0,0.0
9,sales_lines,130402,0,0.0


### **6. Clean Text Columns**

In [13]:
def clean_text_columns(df):

    df = df.copy()

    text_columns = df.select_dtypes(
        include=['object']
    ).columns

    for column in text_columns:

        df[column] = df[column].apply(
            lambda x: x.strip()
            if isinstance(x, str)
            else x
        )

        df[column] = df[column].replace(
            r'^\s*$',
            np.nan,
            regex=True
        )

    return df

for name in datasets:

    datasets[name] = clean_text_columns(
        datasets[name]
    )

print("Text columns cleaned.")

Text columns cleaned.


### **7. Detect Date Columns**

In [15]:
def identify_date_columns(df):

    date_columns = []

    for column in df.columns:

        column_lower = column.lower()

        if any(
            keyword in column_lower
            for keyword in [
                'date',
                'datetime',
                'created',
                'updated'
            ]
        ):

            date_columns.append(column)

    return date_columns

for name, df in datasets.items():

    date_columns = identify_date_columns(df)

    print(
        f"{name}: {date_columns}"
    )

branches: []
customers: ['last_purchase_date']
inventory: []
invoices: ['invoice_date', 'due_date']
payments: ['payment_date']
products: ['last_purchase_date']
purchase_headers: ['order_date', 'expected_delivery_date', 'received_date']
purchase_lines: []
sales_headers: ['order_date', 'delivery_date']
sales_lines: []
stock_ledger: ['movement_date']
suppliers: []


### **8. Convert Date Columns**

In [16]:
for name, df in datasets.items():

    date_columns = identify_date_columns(df)

    for column in date_columns:

        datasets[name][column] = pd.to_datetime(
            datasets[name][column],
            errors='coerce'
        )

print("Date conversion completed.")

Date conversion completed.


### **9. Check Negative Values**

In [17]:
negative_values = []

for name, df in datasets.items():

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for column in numeric_columns:

        count_negative = (
            df[column] < 0
        ).sum()

        if count_negative > 0:

            negative_values.append({
                'Dataset': name,
                'Column': column,
                'Negative Values': count_negative
            })

negative_df = pd.DataFrame(
    negative_values
)

if not negative_df.empty:
    display(negative_df)
else:
    print("No negative numeric values found.")

No negative numeric values found.


### **10. Check Unique IDs**

In [18]:
id_columns_summary = []

for name, df in datasets.items():

    id_columns = [
        column
        for column in df.columns
        if column == 'id'
        or column.endswith('_id')
    ]

    for column in id_columns:

        id_columns_summary.append({
            'Dataset': name,
            'Column': column,
            'Rows': len(df),
            'Unique Values': df[column].nunique(
                dropna=True
            ),
            'Missing': df[column].isna().sum(),
            'Is Unique': df[column].is_unique
        })

id_summary_df = pd.DataFrame(
    id_columns_summary
)

display(id_summary_df)

,Dataset,Column,Rows,Unique Values,Missing,Is Unique
0,branches,branch_id,6,6,0,True
1,branches,manager_id,6,6,0,True
2,customers,customer_id,500,500,0,True
3,customers,branch_id,500,6,0,False
4,inventory,product_id,180,30,0,False
5,inventory,branch_id,180,6,0,False
6,invoices,invoice_id,18033,17836,0,False
7,invoices,so_id,18033,18033,0,True
8,invoices,customer_id,18033,500,0,False
9,invoices,branch_id,18033,6,0,False


### **11. Final Cleaning Summary**

In [19]:
cleaning_summary = []

for name, df in datasets.items():

    cleaning_summary.append({

        'Dataset': name,

        'Rows': df.shape[0],

        'Columns': df.shape[1],

        'Missing Cells': int(
            df.isna().sum().sum()
        ),

        'Duplicate Rows': int(
            df.duplicated().sum()
        )

    })

cleaning_summary_df = pd.DataFrame(
    cleaning_summary
)

display(cleaning_summary_df)

,Dataset,Rows,Columns,Missing Cells,Duplicate Rows
0,branches,6,13,0,0
1,customers,500,15,0,0
2,inventory,180,8,0,0
3,invoices,18033,10,0,0
4,payments,19257,5,0,0
5,products,30,23,0,0
6,purchase_headers,24000,10,2370,0
7,purchase_lines,155495,9,0,0
8,sales_headers,20000,11,0,0
9,sales_lines,130402,9,0,0


### **12. Save Cleaned Datasets**

In [20]:
import os

os.makedirs(
    '/content/week-02-cleaned-data',
    exist_ok=True
)

for name, df in datasets.items():

    output_file = (
        f'/content/week-02-cleaned-data/'
        f'{name}_clean.csv'
    )

    df.to_csv(
        output_file,
        index=False
    )

    print(
        f"Saved: {output_file}"
    )

Saved: /content/week-02-cleaned-data/branches_clean.csv
Saved: /content/week-02-cleaned-data/customers_clean.csv
Saved: /content/week-02-cleaned-data/inventory_clean.csv
Saved: /content/week-02-cleaned-data/invoices_clean.csv
Saved: /content/week-02-cleaned-data/payments_clean.csv
Saved: /content/week-02-cleaned-data/products_clean.csv
Saved: /content/week-02-cleaned-data/purchase_headers_clean.csv
Saved: /content/week-02-cleaned-data/purchase_lines_clean.csv
Saved: /content/week-02-cleaned-data/sales_headers_clean.csv
Saved: /content/week-02-cleaned-data/sales_lines_clean.csv
Saved: /content/week-02-cleaned-data/stock_ledger_clean.csv
Saved: /content/week-02-cleaned-data/suppliers_clean.csv
